# 04. Entrenamiento — Vision Transformer (ViT)

Modelo timm: `vit_base_patch16_224` | Input 128×128 | Transfer learning

## 1. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.dataset import create_dataloaders
from src.models.train import train_model
from src.utils.helpers import get_device, set_seed

MODEL_NAME = "vit_tiny_patch16_224"
MODEL_LABEL = "Vision Transformer (ViT)"

DATA_DIR = ROOT / "data"
MODELS_DIR = DATA_DIR / "06_models"

BATCH_SIZE = 128
NUM_WORKERS = 4
MAX_EPOCHS = 25
PATIENCE = 4
LR = 1e-4
SEED = 42

set_seed(SEED)
device = get_device()
print(f"Modelo: {MODEL_LABEL} ({MODEL_NAME})")
print(f"Device: {device}")

c:\Users\luisg\OneDrive\Documentos\Universidad\7MO\IA\peru-illegal-mining-risk-prediction-model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo: Vision Transformer (ViT) (vit_base_patch16_224)
Device: privateuseone:0


## 2. DataLoaders

In [2]:
loaders = create_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

print(f"Train: {len(train_loader.dataset):,} chips")
print(f"Val:   {len(val_loader.dataset):,} chips")
print(f"Test:  {len(test_loader.dataset):,} chips (usar solo en notebook 05)")

Train: 81,938 chips
Val:   14,840 chips
Test:  14,806 chips (usar solo en notebook 05)


## 3. Entrenamiento

In [3]:
summary = train_model(
    model_name=MODEL_NAME,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=MODELS_DIR,
    device=device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr=LR,
)

print(f"\nMejor epoch: {summary['best_epoch']}")
print(f"Mejor val macro F1: {summary['best_val_macro_f1']:.4f}")

c:\Users\luisg\OneDrive\Documentos\Universidad\7MO\IA\peru-illegal-mining-risk-prediction-model\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luisg\.cache\huggingface\hub\models--timm--vit_base_patch16_224.augreg2_in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Entrenando vit_base_patch16_224 en privateuseone:0
Train: 81,938 | Val: 14,840


train:   0%|          | 0/641 [00:00<?, ?it/s]c:\Users\luisg\OneDrive\Documentos\Universidad\7MO\IA\peru-illegal-mining-risk-prediction-model\.venv\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


RuntimeError: Could not allocate tensor with 103101440 bytes. There is not enough GPU video memory available!

## 4. Curvas de entrenamiento

In [ ]:
history = summary["history"]
epochs = [r["epoch"] for r in history["train"]]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, [r["loss"] for r in history["train"]], label="train")
axes[0].plot(epochs, [r["loss"] for r in history["val"]], label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")

axes[1].plot(epochs, [r["accuracy"] for r in history["train"]], label="train")
axes[1].plot(epochs, [r["accuracy"] for r in history["val"]], label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("epoch")

axes[2].plot(epochs, [r["macro_f1"] for r in history["val"]], label="val macro F1", color="tomato")
axes[2].plot(epochs, [r["recall_com_garimpo"] for r in history["val"]], label="val recall com_garimpo")
axes[2].set_title("Val F1 / Recall com_garimpo"); axes[2].legend(); axes[2].set_xlabel("epoch")

plt.suptitle(f"{MODEL_LABEL} — curvas de entrenamiento", y=1.02)
plt.tight_layout()
fig_path = ROOT / "reports" / "figures" / f"{MODEL_NAME}_training_curves.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {fig_path}")

## 5. Resultado

Checkpoint en `data/06_models/vit_base_patch16_224_best.pt`. Evalúa en test desde `05_evaluation.ipynb`.